# Comparison of Bounded vs. Unbounded ZNE Extrapolation Methods

This notebook compares standard (unbounded) zero-noise extrapolation (ZNE) methods with their physically bounded counterparts.

We focus on:
- Numerical stability of the fit
- Consistency between methods
- Sensitivity to noise scale configurations
- Practical behavior on synthetic and hardware-like data

All methods are evaluated on identical inputs to isolate the effect of bounding constraints.

In [1]:
#import unbounded methods
from mitiq.zne.inference import (PolyFactory, ExpFactory, PolyExpFactory)

# import bounded methods
from src.bounded_methods import (bounded_polynomial_extrapolation, bounded_exp_extrapolation,
                                 bounded_polyexp_extrapolation)

## Method Comparison Setup

We define helper functions to systematically compare bounded and unbounded extrapolation methods.

Key aspects of the comparison:
- **Same inputs** are passed to both bounded and unbounded variants
- Failures (e.g., optimization issues) are captured and reported
- Outputs are compared numerically using a tolerance-based check
- Results are aggregated into a pandas DataFrame for structured analysis

This setup enables:
- Pairwise comparison of methods
- Detection of instability (NaNs, failed fits)
- Quantification of agreement between bounded and unbounded models

In [2]:
import warnings
import numpy as np
import pandas as pd


def _compare_vals(a, b, tol=1e-8):
    if not isinstance(a, (int, float, np.floating)) or not isinstance(b, (int, float, np.floating)):
        return "n/a"
    return "same" if np.isclose(a, b, atol=tol, rtol=tol) else "different"


def _format_warnings(caught_warnings):
    if not caught_warnings:
        return None

    return "; ".join(
        f"{w.category.__name__}: {str(w.message)}"
        for w in caught_warnings
    )


def _safe_call(fn, *args, **kwargs):
    with warnings.catch_warnings(record=True) as caught_warnings:
        warnings.simplefilter("always")
        try:
            result = fn(*args, **kwargs)
            warning_msg = _format_warnings(caught_warnings)
            return result, warning_msg
        except Exception as e:
            warning_msg = _format_warnings(caught_warnings)
            error_msg = f"{type(e).__name__}: {e}"
            if warning_msg:
                return None, f"{error_msg}; warnings: {warning_msg}"
            return None, error_msg


def _mae(val, ground_truth):
    if not isinstance(val, (int, float, np.floating)):
        return None
    return abs(val - ground_truth)


def _improvement(unbounded, bounded, ground_truth):
    mu = _mae(unbounded, ground_truth)
    mb = _mae(bounded, ground_truth)
    if mu is None or mb is None:
        return None
    return mu - mb  # > 0 means bounded is better


def _relative_improvement(unbounded, bounded, ground_truth, eps=1e-12):
    mu = _mae(unbounded, ground_truth)
    mb = _mae(bounded, ground_truth)

    if mu is None or mb is None:
        return None

    # Avoid division by zero (or near-zero instability)
    if mu < eps:
        return 0.0 if mb < eps else None

    return (mu - mb) / mu


def compare_extrapolations_df(
        noise_scale_factors,
        expected_values,
        ground_truth=1.0,
        tol=1e-6,
):
    """
    Compare bounded vs. unbounded extrapolation methods and return results as a pandas DataFrame.

    For each model family (polynomial, exponential, polynomial–exponential) and configuration,
    this function:
      1. Executes both unbounded and bounded extrapolation methods.
      2. Catches and records any exceptions without interrupting execution.
      3. Compares outputs within a numerical tolerance.
      4. Computes mean absolute error (MAE) with respect to a ground truth value.
      5. Computes relative improvement of bounded over unbounded:
             rel_improvement = (MAE_unbounded - MAE_bounded) / MAE_unbounded
         (positive values indicate bounded performs better).

    Parameters
    ----------
    noise_scale_factors : array-like
        Sequence of noise scaling factors (λ values), e.g., [1, 2, 3].
        These are passed identically to both bounded and unbounded methods.

    expected_values : array-like
        Measured expectation values corresponding to each noise scale factor.

    ground_truth : float, default=1.0
        Reference (ideal) value used to compute MAE.

    tol : float, default=1e-6
        Numerical tolerance used when comparing bounded vs. unbounded outputs
        via `np.isclose`.

    Returns
    -------
    pandas.DataFrame
        A table where each row corresponds to a single configuration with columns:
            - method : {"polynomial", "exponential", "polyexp"}
            - order : int or None
            - asymptote : float or None
            - unbounded : float or None
            - bounded : float or None
            - err_unbounded : str or None
            - err_bounded : str or None
            - compare : {"same", "different", "n/a"}
            - mae_unbounded : float or None
            - mae_bounded : float or None
            - rel_improvement : float or None

    Notes
    -----
    - If a method fails, its value is set to None and the error message is recorded.
    - MAE and relative improvement are computed only when valid numeric outputs exist.
    - Relative improvement is undefined (None) when MAE_unbounded is zero (or near zero).
    - The returned DataFrame is suitable for further analysis or styled rendering in notebooks.
    """
    rows = []

    def add_row(method, order, asymptote, unbounded, err_u, bounded, err_b):
        rows.append(
            {
                "method": method,
                "order": order,
                "asymptote": asymptote,
                "unbounded": unbounded,
                "bounded": bounded,
                "err_unbounded": err_u,
                "err_bounded": err_b,
                "compare": _compare_vals(unbounded, bounded, tol),
                "mae_unbounded": _mae(unbounded, ground_truth),
                "mae_bounded": _mae(bounded, ground_truth),
                "improvement": _improvement(unbounded, bounded, ground_truth),
                "rel_improvement": _relative_improvement(unbounded, bounded, ground_truth),
            }
        )

    # Polynomial
    for order in [1, 2]:
        unbounded, err_u = _safe_call(
            PolyFactory.extrapolate,
            scale_factors=noise_scale_factors,
            exp_values=expected_values,
            order=order,
        )
        bounded, err_b = _safe_call(
            bounded_polynomial_extrapolation,
            x=noise_scale_factors,
            y=expected_values,
            order=order,
        )
        add_row("polynomial", order, None, unbounded, err_u, bounded, err_b)

    # Exponential
    for asymptote in [0.0, None]:
        unbounded, err_u = _safe_call(
            ExpFactory.extrapolate,
            scale_factors=noise_scale_factors,
            exp_values=expected_values,
            asymptote=asymptote,
        )
        bounded, err_b = _safe_call(
            bounded_exp_extrapolation,
            x=noise_scale_factors,
            y=expected_values,
            asymptote=asymptote,
        )
        add_row("exponential", None, asymptote, unbounded, err_u, bounded, err_b)

    # Polynomial-Exponential
    for asymptote in [0.0, None]:
        for order in [1]:
            unbounded, err_u = _safe_call(
                PolyExpFactory.extrapolate,
                scale_factors=noise_scale_factors,
                exp_values=expected_values,
                order=order,
                asymptote=asymptote,
            )
            bounded, err_b = _safe_call(
                bounded_polyexp_extrapolation,
                x=noise_scale_factors,
                y=expected_values,
                order=order,
                asymptote=asymptote,
            )
            add_row("polyexp", order, asymptote, unbounded, err_u, bounded, err_b)

    df = pd.DataFrame(rows)

    return df

## Synthetic Example

We begin with a simple synthetic dataset with mild noise decay.

Observations to focus on:
- Whether bounded and unbounded methods produce similar estimates
- Whether any method fails or produces unstable values
- Sensitivity of exponential-family methods to limited data (3 points)

Note:
- Polynomial-exponential (unbounded) may emit fitting warnings, indicating numerical instability or poor conditioning
- Such behavior is expected when the model is flexible relative to the number of data points

In [3]:
noise_scale_factors = [1, 2, 3]
expected_values = [0.2, 0.1, 0.05]

compare_extrapolations_df(noise_scale_factors, expected_values)

,method,order,asymptote,unbounded,bounded,err_unbounded,err_bounded,compare,mae_unbounded,mae_bounded,improvement,rel_improvement
0,polynomial,1.0,NaN,0.266667,0.266667,None,None,same,0.733333,0.733333,2.220446e-16,3.027881e-16
1,polynomial,2.0,NaN,0.350000,0.350000,None,None,same,0.650000,0.650000,3.330669e-16,5.124106e-16
2,exponential,NaN,0.0,0.400000,0.400009,None,None,different,0.600000,0.599991,9.266823e-06,1.544471e-05
3,exponential,NaN,NaN,0.400000,0.400003,ExtrapolationWarning: The extrapolation fit ma...,None,different,0.600000,0.599997,3.369981e-06,5.616635e-06
4,polyexp,1.0,0.0,0.400000,0.399994,None,None,different,0.600000,0.600006,-5.509627e-06,-9.182712e-06
5,polyexp,1.0,NaN,0.400000,0.400055,ExtrapolationWarning: The extrapolation fit ma...,None,different,0.600000,0.599945,5.530544e-05,9.217573e-05


## Analysis of Synthetic Results

In this setting, all extrapolation methods successfully converge and produce finite estimates.
However, two unbounded methods emit warnings from the underlying optimizer, indicating potential ill-conditioning of the fitting problem. This is expected given the small number of data points relative to model flexibility.

### Agreement between bounded and unbounded methods
- The results are nearly identical across most method pairs.
- Specifically, **5 out of 6 bounded/unbounded pairs produce numerically equivalent estimates** within tolerance.
- This suggests that, for well-behaved data, unconstrained optimization already yields physically plausible solutions.

### Effect of bounding
- The bounded variants produce **slightly better estimates** compared to their unbounded counterparts.
- However, the magnitude of this improvement is **very small from a practical perspective**.

### Interpretation
- For simple, low-noise scenarios:
  - Bounding has **minimal impact on accuracy**
  - Its primary benefit is **ensuring physical validity and improving numerical robustness**
- The optimizer warnings indicate that:
  - Even when convergence is achieved, the solution may be **numerically fragile**
  - Bounding acts as a mild regularizer, though its effect is limited in this regime

Overall, bounded and unbounded methods behave almost identically under favorable conditions, with only marginal practical differences.

## Hardware Examples

We now evaluate on data derived from GHZ circuits measured on real hardware (or realistic simulations).

Characteristics of this regime:
- Limited number of noise scale points ($\lambda = 1, 1.3, 1.6$)
- Measurement noise and statistical fluctuations
- Potential deviation from ideal exponential decay

These conditions stress-test extrapolation methods, particularly:
- Stability of nonlinear fits
- Sensitivity to small perturbations
- Risk of unphysical extrapolated values

### GHZ, 30 qubits,  $Z_1 Z_2 \cdots Z_{30}$


We will start with GHZ Observables for 30 qubits

In [4]:
# Case_id: ghz_alg_30__Z_10000_shots_seed_42_group_1
noise_scale_factors = [1, 1.3, 1.6]
expected_values = [0.3114, 0.268, 0.2446]

compare_extrapolations_df(noise_scale_factors, expected_values)

,method,order,asymptote,unbounded,bounded,err_unbounded,err_bounded,compare,mae_unbounded,mae_bounded,improvement,rel_improvement
0,polynomial,1.0,NaN,0.419400,0.419400,None,None,same,0.580600,0.580600,4.440892e-16,7.648798e-16
1,polynomial,2.0,NaN,0.600511,0.600511,None,None,same,0.399489,0.399489,4.107825e-15,1.028270e-14
2,exponential,NaN,0.0,0.463533,0.465877,None,None,different,0.536467,0.534123,2.344072e-03,4.369464e-03
3,exponential,NaN,NaN,0.955459,0.440635,ExtrapolationWarning: The extrapolation fit ma...,None,different,0.044541,0.559365,-5.148238e-01,-1.155838e+01
4,polyexp,1.0,0.0,0.463533,0.465856,None,None,different,0.536467,0.534144,2.322942e-03,4.330077e-03
5,polyexp,1.0,NaN,0.955459,0.954834,ExtrapolationWarning: The extrapolation fit ma...,None,different,0.044541,0.045166,-6.252551e-04,-1.403769e-02


As we can see, all models converged (although Mitiq still emits warnings). Bounded model prevails in 4 out fo 6 cases. The improvement is small (in the third digit or less).

### GHZ, 50 qubits,  $Z_1 Z_2 \cdots Z_{50}$

We now move to a harder case: GHZ with 50 qubits and Z-observables. The amount of noise increases in comparison with the 30 qubits case.

In [5]:
# Case_id: ghz_alg_50__Z_10000_shots_seed_42_group_2
noise_scale_factors = [1, 1.3, 1.6]
expected_values = [0.1468, 0.1286, 0.108]

compare_extrapolations_df(noise_scale_factors, expected_values)

,method,order,asymptote,unbounded,bounded,err_unbounded,err_bounded,compare,mae_unbounded,mae_bounded,improvement,rel_improvement
0,polynomial,1.0,NaN,0.211867,0.211867,None,None,same,0.788133,0.788133,1.110223e-16,1.408674e-16
1,polynomial,2.0,NaN,0.190133,0.190133,None,None,same,0.809867,0.809867,3.330669e-16,4.112614e-16
2,exponential,NaN,0.0,0.245448,0.244352,None,None,different,0.754552,0.755648,-1.095459e-03,-1.451800e-03
3,exponential,NaN,NaN,NaN,0.220650,ExtrapolationError: The extrapolation fit fail...,None,n/a,NaN,0.779350,NaN,NaN
4,polyexp,1.0,0.0,0.245448,0.244353,None,None,different,0.754552,0.755647,-1.094831e-03,-1.450967e-03
5,polyexp,1.0,NaN,NaN,0.218076,ExtrapolationError: The extrapolation fit fail...,None,n/a,NaN,0.781924,NaN,NaN


Two of the unbounded models fail to converge, while the bounded models still return valid (physically constrained) estimates.

This highlights improved robustness of bounded methods under increased noise and more challenging fitting conditions.

Let's look at another execution of the same circuit.

In [6]:
# Case_id: ghz_alg_50__Z_10000_shots_seed_42_group_1
noise_scale_factors = [1, 1.3, 1.6]
expected_values = [0.1578, 0.1208, 0.1166]

compare_extrapolations_df(noise_scale_factors, expected_values)

,method,order,asymptote,unbounded,bounded,err_unbounded,err_bounded,compare,mae_unbounded,mae_bounded,improvement,rel_improvement
0,polynomial,1.0,NaN,0.221000,0.221000,None,None,same,0.779000,0.779000,1.110223e-16,1.425190e-16
1,polynomial,2.0,NaN,0.518022,0.518022,None,None,same,0.481978,0.481978,5.218048e-15,1.082633e-14
2,exponential,NaN,0.0,0.258024,0.264757,None,None,different,0.741976,0.735243,6.732964e-03,9.074370e-03
3,exponential,NaN,NaN,59.050759,0.231294,ExtrapolationWarning: The extrapolation fit ma...,None,different,58.050759,0.768706,5.728205e+01,9.867580e-01
4,polyexp,1.0,0.0,0.258024,0.264755,None,None,different,0.741976,0.735245,6.730808e-03,9.071463e-03
5,polyexp,1.0,NaN,59.050759,1.000000,ExtrapolationWarning: The extrapolation fit ma...,None,different,58.050759,0.000000,5.805076e+01,1.000000e+00


In this case, the unbounded models produce unphysical results in two instances (i.e., values outside the valid $[-1,+1]$ range).

The bounded models consistently return physically valid estimates and show improvement across all cases.

This illustrates that bounding becomes more important in noisier regimes, where unconstrained models are more prone to instability and invalid extrapolation.